In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Lambda,Dense,Embedding

In [ ]:
sentences ="""Deep learning (also known as deep structured learning) is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, convolutional neural networks and Transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance"""

In [ ]:
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer  #Converts words into numerical IDs.

# 1. Clean and tokenize text into words instead of characters
#Important: Keras Tokenizer normally assigns IDs based on descending word frequency.
clean_text = re.sub(r'[^A-Za-z0-9\s]+', '', sentences).lower()  # r means raw string
t = Tokenizer()
t.fit_on_texts([clean_text])
sequence = t.texts_to_sequences([clean_text])[0]

# Update vocabulary size based on word count
vocab = len(t.word_index) + 1 #Because word IDs start from 1, and index 0 is reserved for padding.

# 2. Build context-target pairs
cs = 2
es=10
#Every word will be represented by a vector containing 10 numerical values.
contexts = [] #contexts stores the surrounding words.
targets= [] #targets stores the middle word that we want to predict.

for i in range(cs, len(sequence) - cs):
    context = sequence[i-2:i] + sequence[i+1:i+3]
    target = sequence[i]
    contexts.append(context)
    targets.append(target)

X = np.array(contexts)
y = np.array(targets)

# 3. Re-build, compile and fit the model with corrected dimensions
model = Sequential([
    Embedding(input_dim=vocab, output_dim=es),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),  #axis=1 refers to the context-word dimension.
    Dense(32, activation='relu'),
    Dense(vocab, activation='softmax') #output is vocab Because the model must predict one word from the 100 possible vocabulary indexes.
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=10, validation_data=(X, y))

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.0106 - loss: 4.3311 - val_accuracy: 0.0213 - val_loss: 4.3284
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.0213 - loss: 4.3280 - val_accuracy: 0.0319 - val_loss: 4.3262
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0426 - loss: 4.3258 - val_accuracy: 0.0532 - val_loss: 4.3241
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.0532 - loss: 4.3236 - val_accuracy: 0.0532 - val_loss: 4.3219
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0532 - loss: 4.3213 - val_accuracy: 0.0532 - val_loss: 4.3196
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.0532 - loss: 4.3189 - val_accuracy: 0.0532 - val_loss: 4.3172
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.0532 - loss: 4.3166 - val_accuracy: 0.0426 - val_loss: 4.3146
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0426 - loss: 4.3140 - val_accuracy: 0.0426 - val_loss: 4.3119

In [ ]:
loss, acc = model.evaluate(X, y)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0426 - loss: 4.3057


In [ ]:
itw ={}
itw={v:k for k,v in t.word_index.items()}

In [ ]:
# Train the model for more epochs to allow it to fit the small dataset properly
history = model.fit(X, y, epochs=200, verbose=0) # verbose=0 to keep the output clean

# Evaluate and display the updated predictions
print("--- Predictions after 200 more epochs of training ---")
for i in range(min(5, len(X))): #This loop displays predictions for the first five training examples.

    p = model.predict(X[i:i+1], verbose=0) #X[i:i+1]	(1, 4)	One context with batch dimension,because Keras models expect input in batches
    pi = np.argmax(p[0])

    print("context:", [itw.get(j) for j in X[i]])
    print("actual target:", itw.get(y[i]))
    print("predicted word:", itw.get(pi))
    print("-" * 30)

--- Predictions after 200 more epochs of training ---
context: ['deep', 'learning', 'known', 'as']
actual target: also
predicted word: deep
------------------------------
context: ['learning', 'also', 'as', 'deep']
actual target: known
predicted word: deep
------------------------------
context: ['also', 'known', 'deep', 'structured']
actual target: as
predicted word: as
------------------------------
context: ['known', 'as', 'structured', 'learning']
actual target: deep
predicted word: deep
------------------------------
context: ['as', 'deep', 'learning', 'is']
actual target: structured
predicted word: structured
------------------------------
